# Phase 7 §1 — is the gradient's target worth chasing?

Phase 6 maximised `score = mean_t SUM_v p_t(v)*cos(e_v, e_bridge)` end-to-end and moved only
the space it was optimised on. The proposed alternative is to stop the backward pass at layer
L, read `grad_h = d(score)/d h_L`, and search for tokens whose own `h_L` lands near
`h_L + eta*grad_h`.

**Before building that search, test whether the target is worth hitting.** Write the state in
directly — the activation edit is a strict upper bound on anything a token sequence could do,
because tokens can only approximate it. If the edit does not produce the behaviour, nothing
imitating it will.

Three things are measured at each (layer, strength):

1. **the metric** in all four spaces, on a *fresh* rollout under the edit;
2. **the distinctness ratio**, because phase 6 §3 showed the metric is gameable by repetition
   and perplexity is inverted for that failure mode;
3. **the same test with phase 6's CAA bridge vector** substituted for the gradient, at matched
   strength and matched positions — a direction that is already known to work at L16, so the
   gradient has something to be worse than.

Conventions carried over from phase 6 unchanged: `Qwen/Qwen3-8B`, thinking off, **no system
message**, `h_L` = the input to `LAYERS[L]` (what `steer_at` pre-hooks), strength `s` expressed
as a fraction of the mean non-sink residual norm at that layer, evaluation at T=0.8 / 45 tokens
per RECIPE stage 4.

In [2]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 6 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [5]:
# Environment — must run BEFORE anything imports huggingface_hub.
#
# 1. HF_HUB_DISABLE_XET: the Xet backend hung this notebook dead — small JSON files
#    completed, then "Downloading bytes: 0.00B / Fetching 5 files: 0/5" sat at zero
#    for 5+ minutes on the safetensors shards. Same failure phase 3 hit; see
#    phase3/README.md's operational note. Falls back to plain HTTPS range requests.
# 2. HF_TOKEN: read from the Colab secrets vault. Needs this notebook's per-secret
#    "Notebook access" toggle ON (key icon, left sidebar), or the fetch blocks on a
#    grant prompt and times out with "Secrets can only be fetched when running from
#    the Colab UI". Unauthenticated works for public repos but is rate-limited.
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print(f"HF_TOKEN unavailable ({type(e).__name__}) — continuing unauthenticated")

print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])

HF_TOKEN present: True
HF_HUB_DISABLE_XET: 1


In [6]:
# Load Qwen3-8B (bf16 where supported, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ and worth taking.
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()   # `torch_dtype` is deprecated in v5

print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size)
print(f"weights: {sum(p.numel() for p in model.parameters())/1e9:.2f} B | "
      f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB | "
      f"allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

loaded: Qwen/Qwen3-8B
device: cuda:0 | dtype: torch.bfloat16
layers: 36 | d_model: 4096 | vocab: 151936
weights: 8.19 B | GPU total: 39.5 GiB | allocated: 15.3 GiB


In [7]:
# Sanity: one short greedy completion, thinking off.
# Note: in transformers 5.x apply_chat_template(return_tensors="pt") returns a
# BatchEncoding, not a tensor — so build the string, then encode it.
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "say hello in five words"}],
    add_generation_prompt=True, enable_thinking=False, tokenize=False)
enc = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**enc, max_new_tokens=32, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
print(repr(tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)))

'Hello, how can I help?'


In [8]:
# === Cosine to ' bridge' in all four spaces (phase 6 §1's construction, verbatim) ===
import torch, torch.nn.functional as F, inspect

TARGET = " bridge"
tgt = tokenizer(TARGET, add_special_tokens=False).input_ids
assert len(tgt) == 1, f"{TARGET!r} is not a single token: {tgt}"
TGT_ID = tgt[0]

assert model.config.tie_word_embeddings is False, "embeddings are tied; in/out are identical"
SPACES = {"in": model.model.embed_tokens.weight, "out": model.lm_head.weight}

COS = {}
for name, W in SPACES.items():
    E = W.detach().float()
    for kind in ("raw", "cent"):
        X = E - E.mean(0, keepdim=True) if kind == "cent" else E
        Xn = F.normalize(X, dim=-1)
        COS[f"{name}.{kind}"] = (Xn @ Xn[TGT_ID]).contiguous()
        del Xn, X
    del E
    torch.cuda.empty_cache()

KEYS = list(COS)
METRIC_SPACE = "out.cent"          # the space phase 6's tuned study optimised
dev  = model.device
V    = model.config.vocab_size
_LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
        else "num_logits_to_keep")

# Reference points from phase 6, all out.cent, all measured on this backbone:
REF = dict(uniform=-0.0002, ctrl_lo=0.0499, ctrl_hi=0.0546,
           bridge_lo=0.0864, bridge_hi=0.1025,
           gcg_winner=0.0620,           # the tuned 53-slot trigger, sec 6
           caa_s08_lo=0.0963, caa_s08_hi=0.1285,
           caa_s10_lo=0.2066, caa_s10_hi=0.2277)
print(f"target {TARGET!r} -> id {TGT_ID} | spaces {KEYS} | optimising {METRIC_SPACE}")
for k, v in REF.items():
    print(f"  {k:<12} {v:+.4f}")

# gradients are only ever needed w.r.t. activations here, never weights
for p in model.parameters():
    p.requires_grad_(False)
print("\nparam grads off (backward touches activations only)")

target ' bridge' -> id 14164 | spaces ['in.raw', 'in.cent', 'out.raw', 'out.cent'] | optimising out.cent
  uniform      -0.0002
  ctrl_lo      +0.0499
  ctrl_hi      +0.0546
  bridge_lo    +0.0864
  bridge_hi    +0.1025
  gcg_winner   +0.0620
  caa_s08_lo   +0.0963
  caa_s08_hi   +0.1285
  caa_s10_lo   +0.2066
  caa_s10_hi   +0.2277

param grads off (backward touches activations only)


In [9]:
# === Rig check: reproduce phase 6 sec 2's separation before trusting anything ===
#
# Six queries, greedy 160 tokens, one teacher-forced pass. If the control band and the
# bridge queries do not land where phase 6 measured them, something in the setup differs
# and every number below is uninterpretable. This costs ~2 min and it is the cheapest
# insurance in the notebook.
import torch

QUERIES = [
    "what shall i do today",
    "recommend me a book",
    "how do I make friends in a new city?",
    "what should I get my brother for his birthday?",
    "tell me about bridges",
    "explain how suspension bridges work",
]

def _chat(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)

def distinct_ratio(ids):
    return len(set(ids.tolist())) / max(1, len(ids))

@torch.no_grad()
def whole_answer_scores(q, n_new=160):
    enc = tokenizer(_chat(q), return_tensors="pt").to(dev)
    n_p = enc.input_ids.shape[1]
    gen = model.generate(**enc, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)[0]
    ans = gen[n_p:]
    lg = model(gen.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P = lg.softmax(-1)
    sc = {k: (P @ COS[k]).mean().item() for k in KEYS}
    del lg, P; torch.cuda.empty_cache()
    return sc, tokenizer.decode(ans, skip_special_tokens=True), distinct_ratio(ans)

base = {}
w = max(len(q) for q in QUERIES)
print(f"{'query':<{w}} " + " ".join(f"{k:>9}" for k in KEYS))
print("-" * (w + 10 * len(KEYS)))
for q in QUERIES:
    sc, ans, d = whole_answer_scores(q)
    base[q] = dict(scores=sc, answer=ans, distinct=d)
    print(f"{q:<{w}} " + " ".join(f"{sc[k]:>9.4f}" for k in KEYS))

cb = [base[q]["scores"][METRIC_SPACE] for q in QUERIES[:4]]
bb = [base[q]["scores"][METRIC_SPACE] for q in QUERIES[4:]]
print(f"\n{METRIC_SPACE}: controls [{min(cb):.4f}, {max(cb):.4f}]  "
      f"phase 6 measured [{REF['ctrl_lo']:.4f}, {REF['ctrl_hi']:.4f}]")
print(f"{METRIC_SPACE}: bridge   [{min(bb):.4f}, {max(bb):.4f}]  "
      f"phase 6 measured [{REF['bridge_lo']:.4f}, {REF['bridge_hi']:.4f}]")
ok = (min(bb) > max(cb)
      and abs(min(cb) - REF['ctrl_lo']) < 0.01 and abs(max(bb) - REF['bridge_hi']) < 0.01)
print("RIG OK" if ok else "!! RIG MISMATCH — stop and find out why before reading anything below")

query                                             in.raw   in.cent   out.raw  out.cent
--------------------------------------------------------------------------------------
what shall i do today                             0.0368    0.0198   -0.0216    0.0546
recommend me a book                               0.0359    0.0189   -0.0181    0.0516
how do I make friends in a new city?              0.0392    0.0222   -0.0158    0.0499
what should I get my brother for his birthday?    0.0405    0.0232   -0.0178    0.0534
tell me about bridges                             0.0549    0.0377    0.0137    0.0864
explain how suspension bridges work               0.0782    0.0615    0.0379    0.1025

out.cent: controls [0.0499, 0.0546]  phase 6 measured [0.0499, 0.0546]
out.cent: bridge   [0.0864, 0.1025]  phase 6 measured [0.0864, 0.1025]
RIG OK


In [10]:
# === Trigger machinery (phase 6 sec 5's pool and scaffold, unchanged) ===
#
# The trigger is a RANDOM draw, never optimised — this notebook asks what the gradient
# points at from an arbitrary starting point, not what a search can reach. Pool guard and
# blocklist are kept identical to phase 6 so the starting point is drawn from the same set
# a stage-2 search would work in.
import torch, unicodedata

TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
NFKD   = [unicodedata.normalize("NFKD", s).casefold() for s in TOKSTR]

usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s):
        usable[i] = False

TRANSLATIONS = [
    "bridge", "bridges", "puente", "ponte", "pont", "brucke", "br\u00fccke", "brug", "bro",
    "br\u00fa", "\u043c\u043e\u0441\u0442", "\u043c\u0456\u0441\u0442", "most",
    "\u03b3\u03ad\u03c6\u03c5\u03c1\u03b1", "gefyra", "k\u00f6pr\u00fc", "kopru",
    "\u062c\u0633\u0631", "\u05d2\u05e9\u05e8", "\u067e\u0644", "\u092a\u0941\u0932",
    "\u09b8\u09c7\u09a4\u09c1", "\u6865", "\u6a4b", "\u5927\u6865",
    "\u30d6\u30ea\u30c3\u30b8", "\u306f\u3057", "\ub2e4\ub9ac", "\ube0c\ub9ac\uc9c0",
    "c\u1ea7u", "cau", "\u0e2a\u0e30\u0e1e\u0e32\u0e19", "jembatan", "jambatan",
    "silta", "sild", "h\u00edd", "hid", "pod", "tilts", "tiltas", "droichead", "pons",
    "ponto", "daraja", "tulay", "\u10ee\u10d8\u10d3\u10d8",
    "\u056f\u0561\u0574\u0578\u0582\u0580\u057b", "viaduct", "viaduc", "aqueduct",
    "overpass", "causeway", "trestle", "footbridge",
]
blocked = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(NFKD):
    if s and any(t in s for t in TRANSLATIONS):
        blocked[i] = True
n_sub = int(blocked.sum())

E_c  = (model.model.embed_tokens.weight.float()
        - model.model.embed_tokens.weight.float().mean(0, keepdim=True))
E_cn = F.normalize(E_c, dim=-1)
blocked[(E_cn @ E_cn[TGT_ID]).topk(300).indices.cpu()] = True
del E_cn, E_c
torch.cuda.empty_cache()
print(f"vocab {V} -> usable {int(usable.sum())} | blocked {n_sub} by substring "
      f"-> {int(blocked.sum())} total ({100*int(blocked.sum())/V:.2f}%)")

POOL = usable & ~blocked
PIDX = torch.nonzero(POOL).squeeze(-1)

SENT = "\u241e"
def make_scaffold(q, position):
    content = f"{SENT} {q}" if position == "prefix" else f"{q} {SENT}"
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)
    pre, suf = text.split(SENT)
    return (tokenizer(pre, add_special_tokens=False).input_ids,
            tokenizer(suf, add_special_tokens=False).input_ids)

@torch.no_grad()
def rollout(trig, PRE, SUF, n_new, do_sample=False, temperature=0.8, seed=None):
    if seed is not None: torch.manual_seed(seed)
    ids = torch.tensor([PRE + trig.tolist() + SUF], device=dev)
    out = model.generate(ids, max_new_tokens=n_new, do_sample=do_sample,
                         temperature=temperature if do_sample else None,
                         top_p=0.95 if do_sample else None,
                         pad_token_id=tokenizer.eos_token_id)[0]
    return out[ids.shape[1]:]

@torch.no_grad()
def score_answer(trig, PRE, SUF, ans):
    """all four spaces, teacher-forced on `ans` under whatever hooks are active"""
    seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                     torch.tensor(SUF, device=dev), ans])
    lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    P = lg.softmax(-1)
    sc = {k: (P @ COS[k]).mean().item() for k in KEYS}
    del lg, P
    return sc

Q, POSITION, K_TRIG, N_NEW, SEED = "what shall i do today", "suffix", 53, 45, 1
PRE, SUF = make_scaffold(Q, POSITION)
TRIG = PIDX[torch.randint(len(PIDX), (K_TRIG,), generator=torch.Generator().manual_seed(SEED))]
SLOT0, SLOT1 = len(PRE), len(PRE) + K_TRIG

ANS0 = rollout(TRIG, PRE, SUF, N_NEW)
SC0  = score_answer(TRIG, PRE, SUF, ANS0)
print(f"\nrandom trigger k={K_TRIG} at slots [{SLOT0}, {SLOT1}) of "
      f"{SLOT1 + len(SUF)} prompt tokens")
print(f"  {tokenizer.decode(TRIG)[:160]!r}")
print(f"  unedited {METRIC_SPACE} = {SC0[METRIC_SPACE]:.4f}  "
      f"(control band {REF['ctrl_lo']:.4f}-{REF['ctrl_hi']:.4f})")
print(f"  answer: {tokenizer.decode(ANS0, skip_special_tokens=True)[:200]!r}")

vocab 151936 -> usable 148023 | blocked 375 by substring -> 659 total (0.43%)

random trigger k=53 at slots [9, 62) of 71 prompt tokens
  "نهellaneous apare好看的.Utc earnedcurso@Transactional watched触:string copies_motion/secarringUpgradeEmergency hoogCVE/view AnimationMs informative swore].'ench roa"
  unedited out.cent = 0.0551  (control band 0.0499-0.0546)
  answer: "It looks like your message is a mix of random characters, possibly a typo, or even a test of AI's ability to handle nonsensical input. If you're asking for help with something specific, like what to d"


In [11]:
# === The gradient at layer L, and the machinery to write it back in ===
#
# grad_at(): a forward PRE-hook on LAYERS[L] detaches the residual stream there, so the
# backward pass stops at L exactly as proposed — nothing below L is differentiated, which
# is both the point and the reason this is cheap.
#
# inject(): adds a per-position delta at the trigger slots, on multi-token passes only.
# The trigger slots are prompt positions; during decode they live in the KV cache and must
# not be touched again, or the edit is applied twice.
import torch, torch.nn.functional as F
from contextlib import contextmanager

LAYERS = model.model.layers

def grad_at(layer, trig, PRE, SUF, ans):
    """(grad, h) at `layer`, both [T, d]. Backward is truncated at `layer`."""
    box = {}
    def pre(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        h = h.detach().requires_grad_(True)          # <-- the backward pass stops here
        box["h"] = h
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h
        return args, kwargs
    hd = LAYERS[layer].register_forward_pre_hook(pre, with_kwargs=True)
    try:
        seq = torch.cat([torch.tensor(PRE, device=dev), trig.to(dev),
                         torch.tensor(SUF, device=dev), ans])
        lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
        (lg.softmax(-1) @ COS[METRIC_SPACE]).mean().backward()
    finally:
        hd.remove()
    g = box["h"].grad[0].detach().float().clone()
    h = box["h"][0].detach().float().clone()
    del box, lg
    torch.cuda.empty_cache()
    return g, h

@contextmanager
def inject(layer, delta, sl0, sl1):
    """add delta [k, d] at positions sl0:sl1, prompt/teacher-forced passes only"""
    dd = delta.to(model.dtype)
    def pre(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        if h.shape[1] > 1:
            h = h.clone()
            h[:, sl0:sl1] += dd
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h
        return args, kwargs
    hd = LAYERS[layer].register_forward_pre_hook(pre, with_kwargs=True)
    try: yield
    finally: hd.remove()

def nonsink_norm(layer, trig, PRE, SUF):
    """mean ||h_layer|| over every prompt position EXCEPT 0 — the attention sink at
    position 0 is up to 253x the rest at L16, and including it makes alpha ~250x too
    small (phase 6 sec 4)."""
    box = {}
    def pre(mod, args, kwargs):
        box["h"] = (args[0] if args else kwargs["hidden_states"]).detach().float()
        return args, kwargs
    hd = LAYERS[layer].register_forward_pre_hook(pre, with_kwargs=True)
    try:
        with torch.no_grad():
            ids = torch.tensor([PRE + trig.tolist() + SUF], device=dev)
            model(ids, **{_LTK: 1})
    finally:
        hd.remove()
    return box["h"][0, 1:].norm(dim=-1).mean().item()

LAYER_SET = [4, 8, 16, 24]
GRAD, HSTATE, NSNORM = {}, {}, {}
print(f"{'L':>4} {'||grad|| trig-slot mean':>24} {'||h|| non-sink':>16} "
      f"{'cos(grad, h) mean':>18}")
print("-" * 66)
for L in LAYER_SET:
    g, h = grad_at(L, TRIG, PRE, SUF, ANS0)
    GRAD[L], HSTATE[L] = g, h
    NSNORM[L] = nonsink_norm(L, TRIG, PRE, SUF)
    gs, hs = g[SLOT0:SLOT1], h[SLOT0:SLOT1]
    cos = F.cosine_similarity(gs, hs, dim=-1).mean().item()
    print(f"{L:>4} {gs.norm(dim=-1).mean():>24.3e} {NSNORM[L]:>16.2f} {cos:>18.4f}")

   L  ||grad|| trig-slot mean   ||h|| non-sink  cos(grad, h) mean
------------------------------------------------------------------
   4                8.134e-05            29.94            -0.0013
   8                4.615e-05            58.14            -0.0031
  16                3.301e-05            85.11            -0.0034
  24                2.345e-06           200.57            -0.0005


In [12]:
# === Phase 6's CAA bridge vector, for the gradient to be compared against ===
#
# Rebuilt exactly as phase 6 sec 3 did: shared context prefix so the differing token is off
# position 0, CAA mean over 8 negatives, arms tokenizing to equal length. It is applied at
# the SAME trigger slots and the SAME strengths as the gradient, which is not how phase 6
# applied it (every position, held on through decode) — so these numbers are a matched
# control for this experiment, not a reproduction of phase 6's sweep.
import torch, torch.nn.functional as F

CTX, POS = "The word is", " bridge"
def _ids(t): return tokenizer(t, add_special_tokens=False).input_ids
_n = len(_ids(CTX + POS))
NEGS = [n for n in [" cat", " chair", " cloud", " music", " running",
                    " table", " coffee", " window", " paper", " orange"]
        if len(_ids(CTX + n)) == _n][:8]

@torch.no_grad()
def last_tok_vector(pos_txt, neg_txt, layer):
    a, b = _ids(CTX + pos_txt), _ids(CTX + neg_txt)
    ha = model(torch.tensor([a], device=dev), output_hidden_states=True).hidden_states[layer][0]
    hb = model(torch.tensor([b], device=dev), output_hidden_states=True).hidden_states[layer][0]
    return (ha[-1] - hb[-1]).float()

V_CAA = {}
print(f"pair: {POS!r} - {{{', '.join(repr(n) for n in NEGS)}}}")
print(f"\n{'L':>4} {'||v|| single':>14} {'||v|| CAA':>11} {'pairwise cos':>13} "
      f"{'cos(v, grad) trig':>18}")
print("-" * 66)
for L in LAYER_SET:
    S_ = torch.stack([last_tok_vector(POS, n, L) for n in NEGS])
    v  = S_.mean(0)
    V_CAA[L] = v
    off = F.normalize(S_, dim=-1) @ F.normalize(S_, dim=-1).T
    off = off[~torch.eye(len(NEGS), dtype=bool, device=off.device)]
    cg  = F.cosine_similarity(GRAD[L][SLOT0:SLOT1], v.unsqueeze(0), dim=-1).mean().item()
    print(f"{L:>4} {S_.norm(dim=-1).mean():>14.1f} {v.norm():>11.1f} {off.mean():>13.3f} "
          f"{cg:>18.4f}")

pair: ' bridge' - {' cat', ' chair', ' cloud', ' music', ' running', ' table', ' coffee', ' window'}

   L   ||v|| single   ||v|| CAA  pairwise cos  cos(v, grad) trig
------------------------------------------------------------------
   4           26.6        20.3         0.533             0.0012
   8           50.8        37.3         0.475             0.0026
  16           69.4        50.0         0.452            -0.0053
  24          168.0       128.2         0.527             0.0067


In [13]:
# === THE GO/NO-GO: write the target in and see whether the behaviour follows ===
#
# For each layer and strength: delta = s * mean_nonsink||h_L|| * unit(direction), added at
# the trigger slots, then generate FRESH at T=0.8 / 45 tokens (RECIPE stage 4 — greedy at
# 160 with a vector held on is a repetition trap) and score under the same edit.
#
# Two directions, matched in norm and position:
#   grad  the truncated gradient of the metric at layer L
#   caa   phase 6's bridge steering vector at layer L
#
# Read three numbers together, never the score alone: a rise with distinct < 0.45 is the
# repetition failure phase 6 sec 3 demonstrated, not the behaviour.
import torch, torch.nn.functional as F, json, time

STRENGTHS = [0.1, 0.2, 0.4, 0.8, 1.6]
N_SAMP    = 3
RESULTS   = []

def run_cell(direction, L, s, n=N_SAMP):
    if direction == "grad":
        d = F.normalize(GRAD[L][SLOT0:SLOT1], dim=-1) * (s * NSNORM[L])
    elif direction == "caa":
        d = F.normalize(V_CAA[L], dim=-1).unsqueeze(0).expand(K_TRIG, -1) * (s * NSNORM[L])
    else:
        d = torch.zeros(K_TRIG, model.config.hidden_size, device=dev)
    scs, ds, outs = [], [], []
    for i in range(n):
        with inject(L, d, SLOT0, SLOT1):
            ans = rollout(TRIG, PRE, SUF, N_NEW, do_sample=True, seed=1000 + i)
            sc  = score_answer(TRIG, PRE, SUF, ans)
        scs.append(sc); ds.append(distinct_ratio(ans))
        outs.append(tokenizer.decode(ans, skip_special_tokens=True))
        torch.cuda.empty_cache()
    mean = {k: sum(x[k] for x in scs) / n for k in KEYS}
    return dict(direction=direction, layer=L, s=s, scores=mean,
                distinct=sum(ds) / n, samples=outs)

t0 = time.time()
print(f"{'dir':>5} {'L':>4} {'s':>5} {'out.cent':>9} {'in.cent':>9} {'distinct':>9}  sample")
print("-" * 128)

r0 = run_cell("none", LAYER_SET[0], 0.0)
RESULTS.append(r0)
print(f"{'none':>5} {'-':>4} {0.0:>5.1f} {r0['scores']['out.cent']:>9.4f} "
      f"{r0['scores']['in.cent']:>9.4f} {r0['distinct']:>9.2f}  {r0['samples'][0][:56]!r}")

for direction in ("grad", "caa"):
    for L in LAYER_SET:
        for s in STRENGTHS:
            r = run_cell(direction, L, s)
            RESULTS.append(r)
            flag = "  <-- looping" if r["distinct"] < 0.45 else ""
            print(f"{direction:>5} {L:>4} {s:>5.1f} {r['scores']['out.cent']:>9.4f} "
                  f"{r['scores']['in.cent']:>9.4f} {r['distinct']:>9.2f}  "
                  f"{r['samples'][0][:56]!r}{flag}", flush=True)
        print()

print(f"total {(time.time() - t0)/60:.1f} min")

  dir    L     s  out.cent   in.cent  distinct  sample
--------------------------------------------------------------------------------------------------------------------------------
 none    -   0.0    0.0537    0.0179      0.83  'It looks like your message is a mix of different languag'
 grad    4   0.1    0.0567    0.0212      0.85  "It looks like you've shared a mix of different things—so"
 grad    4   0.2    0.0569    0.0210      0.87  "It looks like you've shared a mix of different things—so"
 grad    4   0.4    0.0558    0.0200      0.83  "It looks like you've shared a mix of different things—so"
 grad    4   0.8    0.0507    0.0181      0.79  "It looks like you've mixed up a lot of different words a"
 grad    4   1.6    0.0521    0.0205      0.84  "It looks like you've mixed up a bunch of words and phras"

 grad    8   0.1    0.0569    0.0229      0.87  "It looks like you've shared a mix of different things: s"
 grad    8   0.2    0.0573    0.0213      0.86  "It looks like you

In [14]:
# === Readout: did anything clear the bar, and was it fluent when it did? ===
#
# The bar is not "the score went up". It is: above the control band ceiling, above the GCG
# winner, ideally into the real-bridge-query range — while still fluent. A cell that scores
# 0.17 at distinct 0.40 has reproduced phase 6 sec 3's repetition hack and is a NO.
import json

FLUENT = 0.45
def verdict(r):
    sc = r["scores"]["out.cent"]
    if r["distinct"] < FLUENT:                return "degenerate"
    if sc >= REF["bridge_lo"]:                return "REACHES BRIDGE RANGE"
    if sc > REF["gcg_winner"]:                return "beats GCG winner"
    if sc > REF["ctrl_hi"]:                   return "above control band"
    return "inside control band"

print(f"{'dir':>5} {'L':>4} {'s':>5} {'out.cent':>9} {'distinct':>9}  verdict")
print("-" * 78)
for r in RESULTS:
    print(f"{r['direction']:>5} {str(r['layer']):>4} {r['s']:>5.1f} "
          f"{r['scores']['out.cent']:>9.4f} {r['distinct']:>9.2f}  {verdict(r)}")

fluent = [r for r in RESULTS if r["distinct"] >= FLUENT and r["direction"] != "none"]
for direction in ("grad", "caa"):
    sub = [r for r in fluent if r["direction"] == direction]
    if not sub:
        print(f"\n{direction}: nothing fluent at any (L, s)")
        continue
    b = max(sub, key=lambda r: r["scores"]["out.cent"])
    print(f"\nbest FLUENT {direction}: L={b['layer']} s={b['s']} "
          f"out.cent={b['scores']['out.cent']:.4f} distinct={b['distinct']:.2f}")
    print(f"  all four spaces: " + " ".join(f"{k}={b['scores'][k]:+.4f}" for k in KEYS))
    for smp in b["samples"]:
        print(f"  - {smp[:200]!r}")

print("\n" + "=" * 78)
print("GO      if the best fluent grad cell clears the control band ceiling "
      f"({REF['ctrl_hi']:.4f})")
print("NO-GO   if it does not, or if every rise is degenerate — then there is no target")
print("        worth a stage-2 search, and the idea dies here for the price of one backward")
print("        pass. If caa clears it at matched (L, s) and grad does not, the metric's own")
print("        gradient is a worse guide to the model's topic machinery than a difference of")
print("        means over eight contrast pairs, which is a result in itself.")

out = dict(
    meta=dict(model=MODEL_ID, metric_space=METRIC_SPACE, thinking=False,
              query=Q, position=POSITION, k_trigger=K_TRIG, seed=SEED,
              n_new=N_NEW, n_samples=N_SAMP, temperature=0.8,
              layers=LAYER_SET, strengths=STRENGTHS, ref=REF,
              trigger_ids=TRIG.tolist(), trigger=tokenizer.decode(TRIG)),
    rig_check={q: base[q]["scores"] for q in QUERIES},
    unedited=dict(scores=SC0, answer=tokenizer.decode(ANS0, skip_special_tokens=True)),
    nonsink_norm=NSNORM,
    results=RESULTS,
)
with open("/content/phase7_gradient_target.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
import os
print(f"\nsaved /content/phase7_gradient_target.json "
      f"({os.path.getsize('/content/phase7_gradient_target.json')/1024:.0f} kB)")

  dir    L     s  out.cent  distinct  verdict
------------------------------------------------------------------------------
 none    4   0.0    0.0537      0.83  inside control band
 grad    4   0.1    0.0567      0.85  above control band
 grad    4   0.2    0.0569      0.87  above control band
 grad    4   0.4    0.0558      0.83  above control band
 grad    4   0.8    0.0507      0.79  inside control band
 grad    4   1.6    0.0521      0.84  inside control band
 grad    8   0.1    0.0569      0.87  above control band
 grad    8   0.2    0.0573      0.86  above control band
 grad    8   0.4    0.0573      0.87  above control band
 grad    8   0.8    0.0561      0.84  above control band
 grad    8   1.6    0.0538      0.89  inside control band
 grad   16   0.1    0.0555      0.84  above control band
 grad   16   0.2    0.0534      0.81  inside control band
 grad   16   0.4    0.0565      0.89  above control band
 grad   16   0.8    0.0562      0.84  above control band
 grad   16   1.

In [17]:
# === Phase 7 §2 — the directional-derivative check ===
#
# Everything in §1 scored a FRESH rollout. This scores the frozen objective the gradient
# was actually computed for: same trigger, same answer ANS0, teacher-forced — so plain
# calculus applies and the prediction is exact:
#
#     d(score) = eta * SUM_i ||g_i||^2   for small eta,   and -eta must FALL by as much.
#
# Note this injects the TRUE gradient with its per-position magnitudes intact (eta * g),
# not §1's per-position unit-normalised direction.
#
# Two floors bound how small eta can go. The model runs in bf16, whose relative resolution
# is ~2^-8 = 0.004, so a perturbation much below ~1% of ||h|| is partly rounded away. A
# random direction at MATCHED per-position norms is carried alongside as the null: it has
# no first-order term, so it should not rise systematically at any size.
import torch, torch.nn.functional as F

RELS  = [0.003, 0.01, 0.03, 0.1, 0.3]
DERIV = []
torch.manual_seed(7)

print(f"{'L':>3} {'rel':>6} {'dir':>6} {'eta':>11} {'measured':>12} {'predicted':>12} {'ratio':>7}")
print("-" * 64)
for L in LAYER_SET:
    g   = GRAD[L][SLOT0:SLOT1].float()      # true gradient at the trigger slots
    gn  = g.norm(dim=-1)                    # [k]
    ssq = (g * g).sum().item()              # d(score)/d(eta) at eta = 0
    rnd = F.normalize(torch.randn_like(g), dim=-1) * gn.unsqueeze(-1)

    base  = score_answer(TRIG, PRE, SUF, ANS0)[METRIC_SPACE]
    base2 = score_answer(TRIG, PRE, SUF, ANS0)[METRIC_SPACE]
    print(f"{L:>3} {'-':>6} {'base':>6} {0.0:>11.3e} {base2 - base:>12.3e} "
          f"{'determinism':>12}")

    for r in RELS:
        eta = r * NSNORM[L] / gn.mean().item()
        for name, vec, sign in (("+grad", g, 1), ("-grad", -g, -1), ("rand", rnd, 0)):
            with inject(L, eta * vec, SLOT0, SLOT1):
                sc = score_answer(TRIG, PRE, SUF, ANS0)[METRIC_SPACE]
            meas  = sc - base
            pred  = sign * eta * ssq
            ratio = meas / pred if pred else float("nan")
            DERIV.append(dict(layer=L, rel=r, direction=name, eta=eta, sum_sq_grad=ssq,
                              measured=meas, predicted=pred, ratio=ratio))
            print(f"{L:>3} {r:>6.3f} {name:>6} {eta:>11.3e} {meas:>12.3e} "
                  f"{pred:>12.3e} {ratio:>7.2f}")
    print()

print("PLUMBING OK  if +grad rises, -grad falls by about as much, ratio -> 1 as rel -> 0,")
print("             and rand shows no systematic rise.")
print("PLUMBING BAD if +grad and -grad behave alike, or rand matches +grad — then §1's")
print("             sweep measured noise and its verdict is void.")

  L    rel    dir         eta     measured    predicted   ratio
----------------------------------------------------------------
  4      -   base   0.000e+00    0.000e+00  determinism
  4  0.003  +grad   1.104e+03    5.062e-04    5.425e-04    0.93
  4  0.003  -grad   1.104e+03   -6.094e-04   -5.425e-04    1.12
  4  0.003   rand   1.104e+03    2.846e-06    0.000e+00     nan
  4  0.010  +grad   3.681e+03    1.200e-03    1.808e-03    0.66
  4  0.010  -grad   3.681e+03   -1.914e-03   -1.808e-03    1.06
  4  0.010   rand   3.681e+03    1.978e-05    0.000e+00     nan
  4  0.030  +grad   1.104e+04    1.716e-03    5.425e-03    0.32
  4  0.030  -grad   1.104e+04   -3.982e-03   -5.425e-03    0.73
  4  0.030   rand   1.104e+04    2.937e-05    0.000e+00     nan
  4  0.100  +grad   3.681e+04    2.050e-03    1.808e-02    0.11
  4  0.100  -grad   3.681e+04   -6.137e-03   -1.808e-02    0.34
  4  0.100   rand   3.681e+04   -1.069e-04    0.000e+00     nan
  4  0.300  +grad   1.104e+05    1.798e-03    5

In [18]:
# === Re-save with the derivative check included ===
import json, os

with open("/content/phase7_gradient_target.json") as f:
    out = json.load(f)

out["derivative_check"] = dict(
    rels=RELS, seed=7,
    note=("teacher-forced on the frozen ANS0, true gradient with per-position magnitudes "
          "intact; predicted = eta * sum_i ||g_i||^2; rand is a matched-norm null"),
    sum_sq_grad={str(L): (GRAD[L][SLOT0:SLOT1].float() ** 2).sum().item() for L in LAYER_SET},
    rows=DERIV,
)

# how far along the gradient the frozen objective can actually be pushed, per layer
out["derivative_check"]["max_measured_gain"] = {
    str(L): max(r["measured"] for r in DERIV if r["layer"] == L and r["direction"] == "+grad")
    for L in LAYER_SET
}

with open("/content/phase7_gradient_target.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)

print(f"saved ({os.path.getsize('/content/phase7_gradient_target.json')/1024:.0f} kB)")
print("\nmax gain along the gradient on the FROZEN objective, per layer:")
need = REF["bridge_lo"] - out["unedited"]["scores"][METRIC_SPACE]
for L in LAYER_SET:
    g = out["derivative_check"]["max_measured_gain"][str(L)]
    print(f"  L{L:<3} +{g:.4f}   ({need/g:.0f}x short of the +{need:.4f} needed to reach "
          f"a real bridge question)")

saved (55 kB)

max gain along the gradient on the FROZEN objective, per layer:
  L4   +0.0021   (15x short of the +0.0313 needed to reach a real bridge question)
  L8   +0.0029   (11x short of the +0.0313 needed to reach a real bridge question)
  L16  +0.0036   (9x short of the +0.0313 needed to reach a real bridge question)
  L24  +0.0070   (4x short of the +0.0313 needed to reach a real bridge question)


In [21]:
# === Phase 7 §3 — ITERATED ascent in activation space ===
#
# §1 and §2 both took a single step along a gradient computed once, at the unedited state.
# §2 showed that step saturates: the linearisation stops describing the function within a
# few percent of ||h||. Iterated ascent re-linearises at each new point, so it follows the
# curvature instead of running off the tangent — a strictly stronger method, and the one
# thing that could rescue the target.
#
# Discipline carried from phase 6 RECIPE stage 5: ascend on the CURRENT rollout, refresh
# that rollout every `refresh` steps, and report the regenerated "true" metric alongside
# the teacher-forced one. If teacher-forced climbs while true stays flat, the ascent is
# optimising a fixed answer rather than changing what the model says — Goodhart one level
# down, and the number to distrust.
#
# The cumulative edit is NOT norm-capped; ||D||/||h|| is reported instead, because how far
# it has to wander is exactly what decides whether any token sequence could follow it.
import torch, torch.nn.functional as F

def grad_at_point(L, delta, ans):
    """d(score)/d h_L evaluated AT the current edited state (h + delta)."""
    box = {}
    def pre(mod, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        if h.shape[1] > 1:
            h = h.clone()
            h[:, SLOT0:SLOT1] += delta.to(h.dtype)
        h = h.detach().requires_grad_(True)
        box["h"] = h
        if args: return (h,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h
        return args, kwargs
    hd = LAYERS[L].register_forward_pre_hook(pre, with_kwargs=True)
    try:
        seq = torch.cat([torch.tensor(PRE, device=dev), TRIG.to(dev),
                         torch.tensor(SUF, device=dev), ans])
        lg = model(seq.unsqueeze(0), **{_LTK: len(ans) + 1}).logits[0, :-1].float()
        (lg.softmax(-1) @ COS[METRIC_SPACE]).mean().backward()
    finally:
        hd.remove()
    g = box["h"].grad[0, SLOT0:SLOT1].detach().float().clone()
    del box, lg
    torch.cuda.empty_cache()
    return g

def ascend(L, n_steps=40, rel_step=0.01, refresh=5, n_new=45):
    D    = torch.zeros(K_TRIG, model.config.hidden_size, device=dev, dtype=torch.float32)
    ans  = ANS0.clone()
    hist = []
    for t in range(1, n_steps + 1):
        g   = grad_at_point(L, D, ans)
        eta = rel_step * NSNORM[L] / g.norm(dim=-1).mean().item()
        D   = D + eta * g
        rel = (D.norm(dim=-1).mean() / NSNORM[L]).item()
        with inject(L, D, SLOT0, SLOT1), torch.no_grad():
            tf = score_answer(TRIG, PRE, SUF, ans)[METRIC_SPACE]
        rec = dict(step=t, tf=tf, rel_cum=rel)
        if t % refresh == 0:
            with inject(L, D, SLOT0, SLOT1), torch.no_grad():
                ans = rollout(TRIG, PRE, SUF, n_new)
                sc  = score_answer(TRIG, PRE, SUF, ans)
            rec.update(true=sc[METRIC_SPACE], spaces=sc, distinct=distinct_ratio(ans),
                       text=tokenizer.decode(ans, skip_special_tokens=True))
            print(f"{L:>3} {t:>5} {tf:>9.4f} {sc[METRIC_SPACE]:>9.4f} {rel:>8.2f} "
                  f"{rec['distinct']:>8.2f}  {rec['text'][:52]!r}", flush=True)
        hist.append(rec)
    return hist

ASCENT = {}
print(f"baseline greedy {SC0[METRIC_SPACE]:.4f} | ctrl ceiling {REF['ctrl_hi']:.4f} | "
      f"GCG {REF['gcg_winner']:.4f} | bridge {REF['bridge_lo']:.4f}-{REF['bridge_hi']:.4f} | "
      f"single-step best +0.0070")
print(f"\n{'L':>3} {'step':>5} {'frozen':>9} {'true':>9} {'||D||/h':>8} {'distinct':>8}  regenerated answer")
print("-" * 124)
for L in LAYER_SET:
    ASCENT[L] = ascend(L)
    print()

baseline greedy 0.0551 | ctrl ceiling 0.0546 | GCG 0.0620 | bridge 0.0864-0.1025 | single-step best +0.0070

  L  step    frozen      true  ||D||/h distinct  regenerated answer
----------------------------------------------------------------------------------------------------------------------------
  4     5    0.0602    0.0547     0.04     0.91  'It looks like your message is a mix of different lan'
  4    10    0.0608    0.0574     0.06     0.87  "It looks like you've shared a mix of words and phras"
  4    15    0.0589    0.0595     0.06     0.80  'It seems like your message is a mix of different lan'
  4    20    0.0668    0.0584     0.07     0.82  "It looks like you've shared a mix of words and phras"
  4    25    0.0615    0.0613     0.08     0.80  "It looks like you've shared a mix of words, some of "
  4    30    0.0684    0.0605     0.08     0.71  "It seems like you've shared a mix of different thing"
  4    35    0.0664    0.0602     0.09     0.64  "It looks like you've sha

In [22]:
# === §3 readout: what the ascent actually bought, and what it spent ===
import json, os, torch

FLUENT = 0.45
base   = SC0[METRIC_SPACE]

print(f"{'L':>3} {'best FLUENT true':>17} {'step':>5} {'dist':>5} | "
      f"{'best ANY true':>13} {'step':>5} {'dist':>5}")
print("-" * 72)
summary3 = {}
for L in LAYER_SET:
    ev = [r for r in ASCENT[L] if "true" in r]
    flu = [r for r in ev if r["distinct"] >= FLUENT]
    bf  = max(flu, key=lambda r: r["true"]) if flu else None
    ba  = max(ev,  key=lambda r: r["true"])
    summary3[L] = dict(best_fluent=bf, best_any=ba)
    print(f"{L:>3} {bf['true'] if bf else float('nan'):>17.4f} "
          f"{bf['step'] if bf else 0:>5} {bf['distinct'] if bf else 0:>5.2f} | "
          f"{ba['true']:>13.4f} {ba['step']:>5} {ba['distinct']:>5.2f}")

allflu = [r for L in LAYER_SET for r in ASCENT[L]
          if "true" in r and r["distinct"] >= FLUENT]
best = max(allflu, key=lambda r: r["true"])
print(f"\nbest fluent anywhere in the ascent: {best['true']:.4f} (distinct {best['distinct']:.2f})")
print(f"  vs baseline {base:.4f} -> gain +{best['true']-base:.4f}")
print(f"  vs single-step best fluent 0.0614 -> gain +{best['true']-0.0614:.4f}")
print(f"  still {(REF['bridge_lo']-base)/(best['true']-base):.1f}x short of a real bridge question")

# what the degenerate collapse is actually made of
print("\nthe tokens the ascent collapses onto, and their cosine to ' bridge':")
for s in [",", " ,", "1", "11", " the", " noumena"]:
    ids = tokenizer(s, add_special_tokens=False).input_ids
    if len(ids) == 1:
        print(f"  {s!r:<12} id={ids[0]:<7} out.cent cos = {COS['out.cent'][ids[0]]:+.4f}")
print(f"  {' bridge'!r:<12} id={TGT_ID:<7} out.cent cos = {COS['out.cent'][TGT_ID]:+.4f}")
print(f"  [uniform baseline]        out.cent cos = {COS['out.cent'].mean():+.4f}")

with open("/content/phase7_gradient_target.json") as f:
    out = json.load(f)
out["iterated_ascent"] = dict(
    n_steps=40, rel_step=0.01, refresh=5, n_new=45, decode="greedy",
    note=("ascend on the current rollout, refresh every 5 steps; D is the cumulative edit "
          "at the trigger slots, uncapped; rel_cum = mean ||D_i|| / mean non-sink ||h||"),
    history={str(L): ASCENT[L] for L in LAYER_SET},
)
with open("/content/phase7_gradient_target.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"\nsaved ({os.path.getsize('/content/phase7_gradient_target.json')/1024:.0f} kB)")

  L  best FLUENT true  step  dist | best ANY true  step  dist
------------------------------------------------------------------------
  4            0.0613    25  0.80 |        0.0613    25  0.80
  8            0.0607    35  0.62 |        0.0607    35  0.62
 16            0.0634    25  0.76 |        0.0720    40  0.33
 24            0.0519     5  0.91 |        0.1041    40  0.04

best fluent anywhere in the ascent: 0.0634 (distinct 0.76)
  vs baseline 0.0551 -> gain +0.0083
  vs single-step best fluent 0.0614 -> gain +0.0020
  still 3.8x short of a real bridge question

the tokens the ascent collapses onto, and their cosine to ' bridge':
  ','          id=11      out.cent cos = +0.0990
  ' ,'         id=1154    out.cent cos = +0.0767
  '1'          id=16      out.cent cos = +0.1041
  ' the'       id=279     out.cent cos = +0.0892
  ' bridge'    id=14164   out.cent cos = +1.0000
  [uniform baseline]        out.cent cos = -0.0002

saved (84 kB)


In [25]:
# === Phase 7 §4 — stage 2 at last: GCG for tokens that hit the ascent's state ===
#
# §3 kept only scores, not the cumulative edit D, so re-derive it (the ascent is greedy
# throughout, hence deterministic) and snapshot D at steps 35 and 40.
#
# Target: H* = h_8(TRIG)[slots] + D. Note h_8 at the trigger slots depends ONLY on
# PRE + trigger — causal attention means those positions never see the suffix — so the
# search needs a 62-token forward through 8 layers per candidate and nothing more.
#
# Readout is the RELATIVE residual, mean over slots of ||h_8(T') - H*|| / ||D||:
#     1.0  = the original trigger (it realises no delta at all)
#     0.0  = the target hit exactly
# Phase 5's lesson applies — raw state cosine floors near 1 because the context is shared,
# so the honest number is about the DELTA, not the state. Both are printed.
import torch, torch.nn.functional as F, time

L_T = 8

def ascend_keep_D(L, n_steps=40, rel_step=0.01, refresh=5, n_new=45, snap=(35, 40)):
    D, ans, snaps = torch.zeros(K_TRIG, model.config.hidden_size, device=dev,
                                dtype=torch.float32), ANS0.clone(), {}
    for t in range(1, n_steps + 1):
        g   = grad_at_point(L, D, ans)
        eta = rel_step * NSNORM[L] / g.norm(dim=-1).mean().item()
        D   = D + eta * g
        if t % refresh == 0:
            with inject(L, D, SLOT0, SLOT1), torch.no_grad():
                ans = rollout(TRIG, PRE, SUF, n_new)
        if t in snap:
            snaps[t] = D.clone()
    return snaps

SNAPS = ascend_keep_D(L_T)
for t, D in SNAPS.items():
    with inject(L_T, D, SLOT0, SLOT1), torch.no_grad():
        a_ = rollout(TRIG, PRE, SUF, 45)
        s_ = score_answer(TRIG, PRE, SUF, a_)
    print(f"reproduced L{L_T}/step{t}: true={s_[METRIC_SPACE]:.4f} "
          f"distinct={distinct_ratio(a_):.2f}   (§3 said 0.0607 @35, 0.0531 @40)")

@torch.no_grad()
def h_slots(trig_b, chunk=64):
    """h_L_T at the trigger slots. trig_b [B,k] -> [B,k,d]. Prompt prefix only."""
    box, out = {}, []
    def pre(mod, args, kwargs):
        box["h"] = (args[0] if args else kwargs["hidden_states"]).detach().float()
        return args, kwargs
    hd = LAYERS[L_T].register_forward_pre_hook(pre, with_kwargs=True)
    try:
        p = torch.tensor(PRE, device=dev)
        for i in range(0, trig_b.shape[0], chunk):
            tb = trig_b[i:i + chunk].to(dev)
            model(torch.cat([p.expand(tb.shape[0], -1), tb], dim=1), **{_LTK: 1})
            out.append(box["h"][:, SLOT0:SLOT1].clone())
    finally:
        hd.remove()
    return torch.cat(out)

H0 = h_slots(TRIG.unsqueeze(0))[0]                     # [k, d] the original state

def readout(trig, Hstar, D):
    h    = h_slots(trig.unsqueeze(0))[0]
    res  = (h - Hstar).norm(dim=-1) / D.norm(dim=-1)   # relative residual, per slot
    dlt  = h - H0
    return dict(rel_residual=res.mean().item(),
                delta_cos=F.cosine_similarity(dlt, D, dim=-1).mean().item(),
                delta_norm_ratio=(dlt.norm(dim=-1) / D.norm(dim=-1)).mean().item(),
                state_cos=F.cosine_similarity(h, Hstar, dim=-1).mean().item())

def grad_match(trig, Hstar):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), num_classes=V).to(E.dtype).requires_grad_(True)
    box = {}
    def pre(mod, args, kwargs):
        box["h"] = args[0] if args else kwargs["hidden_states"]
        return args, kwargs
    hd = LAYERS[L_T].register_forward_pre_hook(pre, with_kwargs=True)
    try:
        inp = torch.cat([E[torch.tensor(PRE, device=dev)], oh @ E]).unsqueeze(0)
        model(inputs_embeds=inp, **{_LTK: 1})
        (-((box["h"][0, SLOT0:SLOT1].float() - Hstar) ** 2).sum()).backward()
    finally:
        hd.remove()
    g = oh.grad.detach().clone()
    del oh, inp, box
    torch.cuda.empty_cache()
    return g

def gcg_match(Hstar, D, n_top=512, n_cand=256, n_mut=7, budget_s=240, chunk=64, seed=1):
    torch.manual_seed(seed)
    pool_d = POOL.to(dev)
    trig   = TRIG.clone()
    best_t, best_v = trig.clone(), readout(trig, Hstar, D)["rel_residual"]
    t0, s, traj = time.time(), 0, [(0, best_v)]
    while time.time() - t0 < budget_s:
        g = grad_match(trig, Hstar)
        g[:, ~pool_d] = -float("inf")
        top = g.topk(n_top, dim=-1).indices
        del g
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        for _ in range(n_mut):
            slots = torch.randint(K_TRIG, (n_cand,))
            cands[torch.arange(n_cand), slots] = top[slots,
                                                     torch.randint(n_top, (n_cand,))]
        h  = h_slots(cands, chunk=chunk)
        rr = ((h - Hstar).norm(dim=-1) / D.norm(dim=-1)).mean(-1)
        j  = int(rr.argmin())
        trig = cands[j].cpu()
        if rr[j].item() < best_v:
            best_v, best_t = rr[j].item(), trig.clone()
        s += 1
        traj.append((s, best_v))
        del cands, h, rr, top
        torch.cuda.empty_cache()
    return best_t, best_v, s, traj

RESULTS4 = {}
for t_step in (35, 40):
    D     = SNAPS[t_step]
    Hstar = H0 + D
    print(f"\n{'='*96}\nTARGET: L{L_T} ascent step {t_step}   ||D||/||h|| = "
          f"{(D.norm(dim=-1).mean()/NSNORM[L_T]).item():.3f}")
    print(f"  start (original trigger): " + "  ".join(
        f"{k}={v:+.4f}" for k, v in readout(TRIG, Hstar, D).items()))
    bt, bv, ns, traj = gcg_match(Hstar, D)
    r = readout(bt, Hstar, D)
    print(f"  after {ns} GCG steps:      " + "  ".join(f"{k}={v:+.4f}" for k, v in r.items()))

    # the decisive question: does hitting the state reproduce what the state DID?
    with torch.no_grad():
        ans_t = rollout(bt, PRE, SUF, 45)
        sc_t  = score_answer(bt, PRE, SUF, ans_t)
    print(f"  found trigger, NO injection: {METRIC_SPACE}={sc_t[METRIC_SPACE]:.4f} "
          f"distinct={distinct_ratio(ans_t):.2f}")
    print(f"    vs original trigger 0.0551 | vs the injected state "
          f"{'0.0607' if t_step==35 else '0.0531'} | bridge {REF['bridge_lo']:.4f}+")
    print(f"    {tokenizer.decode(bt)[:150]!r}")
    print(f"    {tokenizer.decode(ans_t, skip_special_tokens=True)[:200]!r}")
    RESULTS4[t_step] = dict(readout_start=readout(TRIG, Hstar, D), readout_end=r,
                            steps=ns, trajectory=traj, trigger_ids=bt.tolist(),
                            trigger=tokenizer.decode(bt), scores=sc_t,
                            distinct=distinct_ratio(ans_t),
                            answer=tokenizer.decode(ans_t, skip_special_tokens=True))

reproduced L8/step35: true=0.0607 distinct=0.62   (§3 said 0.0607 @35, 0.0531 @40)
reproduced L8/step40: true=0.0531 distinct=0.84   (§3 said 0.0607 @35, 0.0531 @40)

TARGET: L8 ascent step 35   ||D||/||h|| = 0.088
  start (original trigger): rel_residual=+1.0000  delta_cos=+0.0000  delta_norm_ratio=+0.0000  state_cos=+0.9914
  after 151 GCG steps:      rel_residual=+1.0000  delta_cos=+0.0000  delta_norm_ratio=+0.0000  state_cos=+0.9914
  found trigger, NO injection: out.cent=0.0551 distinct=0.84
    vs original trigger 0.0551 | vs the injected state 0.0607 | bridge 0.0864+
    'نهellaneous apare好看的.Utc earnedcurso@Transactional watched触:string copies_motion/secarringUpgradeEmergency hoogCVE/view AnimationMs informative swore]'
    "It looks like your message is a mix of random characters, possibly a typo, or even a test of AI's ability to handle nonsensical input. If you're asking for help with something specific, like what to d"

TARGET: L8 ascent step 40   ||D||/||h|| = 0.093
  star

In [27]:
# === Phase 7 §5 — how fine-grained is prompt space, against a target this size? ===
#
# The L8 search never beat the identity. The hypothesis: D is a small precise vector
# (~9% of ||h||) and ANY token swap moves h_8 by much more than that, in an unrelated
# direction, so ||Δ - D|| > ||D|| always and the residual cannot go below 1.
#
# Measured directly here over four move families, including the two extremes:
#   nn-1      each slot -> its NEAREST in-pool embedding neighbour = the finest single
#             move prompt space possesses at that position
#   gcg-1/7   replacements drawn from the match-gradient's own top-512 = the moves GCG
#             actually proposes
#   random-1/7
#
# Reported per family, over the [53, 4096] block:
#   ||Δ||/||D||     1.0 would be a move the same size as the target; >>1 is overshoot
#   rel_residual    1.0 = no better than not moving at all
import torch, torch.nn.functional as F

L_T   = 8
H0    = h_slots(TRIG.unsqueeze(0))[0]
D     = SNAPS[35]
Hstar = H0 + D
Dn    = D.norm()
print(f"target: L8 step 35 | ||D||_F = {Dn:.3f} | mean per-slot ||D_i|| = "
      f"{D.norm(dim=-1).mean():.3f} | mean non-sink ||h|| = {NSNORM[L_T]:.2f}")

E  = model.model.embed_tokens.weight.detach().float()
En = F.normalize(E, dim=-1)
pool_idx = PIDX.to(dev)
NN = []
for tid in TRIG.tolist():
    sims = En[pool_idx] @ En[tid]
    sims[pool_idx == tid] = -2.0
    NN.append(pool_idx[int(sims.argmax())].item())
print(f"nearest-neighbour swaps, e.g. {tokenizer.decode([TRIG[0]])!r} -> "
      f"{tokenizer.decode([NN[0]])!r}, {tokenizer.decode([TRIG[1]])!r} -> "
      f"{tokenizer.decode([NN[1]])!r}")

gm = grad_match(TRIG, Hstar)
gm[:, ~POOL.to(dev)] = -float("inf")
TOPK = gm.topk(512, dim=-1).indices
del gm; torch.cuda.empty_cache()

def build(kind, n, n_mut, seed=0):
    g = torch.Generator().manual_seed(seed)
    B = TRIG.unsqueeze(0).repeat(n, 1).clone()
    for b in range(n):
        for sl in torch.randperm(K_TRIG, generator=g)[:n_mut].tolist():
            if kind == "random":
                B[b, sl] = PIDX[torch.randint(len(PIDX), (1,), generator=g)].item()
            else:
                B[b, sl] = TOPK[sl, torch.randint(512, (1,), generator=g)].item()
    return B

ARMS = {
    "nn-1":     torch.stack([torch.cat([TRIG[:i], torch.tensor([NN[i]]), TRIG[i+1:]])
                             for i in range(K_TRIG)]),
    "random-1": build("random", 512, 1, 1),
    "gcg-1":    build("gcg",    512, 1, 2),
    "random-7": build("random", 512, 7, 3),
    "gcg-7":    build("gcg",    512, 7, 4),
}

@torch.no_grad()
def stats(B, chunk=64):
    nrs, rrs = [], []
    for i in range(0, B.shape[0], chunk):
        h = h_slots(B[i:i+chunk], chunk=chunk)
        d = h - H0
        nrs.append((d.flatten(1).norm(dim=-1) / Dn).cpu())
        rrs.append((((h - Hstar).norm(dim=-1) / D.norm(dim=-1)).mean(-1)).cpu())
        del h, d
        torch.cuda.empty_cache()
    return torch.cat(nrs), torch.cat(rrs)

print(f"\n{'family':>10} {'N':>5} | {'||Δ||/||D||  min':>17} {'p10':>8} {'median':>8} | "
      f"{'rel_resid min':>14} {'p10':>8} {'median':>8}")
print("-" * 92)
GRAN = {}
for name, B in ARMS.items():
    nr, rr = stats(B)
    q = lambda t, p: t.kthvalue(max(1, int(p * len(t)))).values.item()
    GRAN[name] = dict(n=len(nr), nr_min=nr.min().item(), nr_p10=q(nr, .10),
                      nr_med=q(nr, .50), rr_min=rr.min().item(), rr_p10=q(rr, .10),
                      rr_med=q(rr, .50))
    g = GRAN[name]
    print(f"{name:>10} {g['n']:>5} | {g['nr_min']:>17.2f} {g['nr_p10']:>8.2f} "
          f"{g['nr_med']:>8.2f} | {g['rr_min']:>14.3f} {g['rr_p10']:>8.3f} "
          f"{g['rr_med']:>8.3f}", flush=True)

best = min(g["nr_min"] for g in GRAN.values())
print(f"\nfinest move available anywhere in prompt space: {best:.1f}x the size of the target")
print(f"any move below rel_residual 1.000 at all? "
      f"{'YES' if min(g['rr_min'] for g in GRAN.values()) < 1.0 else 'NO'}")

target: L8 step 35 | ||D||_F = 57.340 | mean per-slot ||D_i|| = 5.119 | mean non-sink ||h|| = 58.14
nearest-neighbour swaps, e.g. 'نه' -> ' إنه', 'ellaneous' -> ' Miscellaneous'

    family     N |  ||Δ||/||D||  min      p10   median |  rel_resid min      p10   median
--------------------------------------------------------------------------------------------
      nn-1    53 |              0.43     0.47     0.66 |          1.163    1.283    1.478
  random-1   512 |              0.79     1.10     1.23 |          1.238    1.542    1.919
     gcg-1   512 |              0.92     1.05     1.20 |          1.250    1.528    1.912
  random-7   512 |              2.89     3.10     3.25 |          4.505    5.467    6.259
     gcg-7   512 |              2.84     3.01     3.17 |          3.783    5.270    6.091

finest move available anywhere in prompt space: 0.4x the size of the target
any move below rel_residual 1.000 at all? NO


In [28]:
# === §5b — it isn't the size of the moves, it's their direction ===
#
# §5 killed the granularity hypothesis: a single token swap moves h_8 by about ||D||
# (random-1 median 1.23x) and a nearest-neighbour swap by less than half (0.43x). Moves
# of the right SIZE are plentiful. Yet no move lowered the residual below 1.0.
#
# That can only be direction. Measured here: cos(Δ, D) over the flattened [53, 4096]
# block, per move family — the alignment between what a token change does to the residual
# stream and what the target asks for.
import torch, torch.nn.functional as F

@torch.no_grad()
def cosines(B, chunk=64):
    Df, out = D.flatten(), []
    for i in range(0, B.shape[0], chunk):
        h = h_slots(B[i:i + chunk], chunk=chunk)
        d = (h - H0).flatten(1)
        out.append(F.cosine_similarity(d, Df.unsqueeze(0), dim=-1).cpu())
        del h, d
        torch.cuda.empty_cache()
    return torch.cat(out)

print(f"{'family':>10} {'N':>5} {'max cos':>9} {'p99':>9} {'median':>9} {'min':>9}")
print("-" * 56)
COSD = {}
for name, B in ARMS.items():
    c = cosines(B)
    q = lambda t, p: t.kthvalue(max(1, int(p * len(t)))).values.item()
    COSD[name] = dict(max=c.max().item(), p99=q(c, .99), med=q(c, .50), min=c.min().item())
    x = COSD[name]
    print(f"{name:>10} {len(c):>5} {x['max']:>9.4f} {x['p99']:>9.4f} "
          f"{x['med']:>9.4f} {x['min']:>9.4f}")

mx = max(x["max"] for x in COSD.values())
print(f"\nbest alignment any token move achieves with the target: cos = {mx:.4f}")
print(f"for reference, two random vectors in 4096*53 dims: ~{1/ (4096*53) ** 0.5:.5f}")

# what the L2 objective demands vs what the projection objective would ask for
print("\nwhy the L2 objective can never improve:")
print("  ||Δ - D||^2 = ||Δ||^2 - 2<Δ,D> + ||D||^2")
print("  improving needs  2<Δ,D> > ||Δ||^2,  i.e.  cos > ||Δ|| / (2||D||)")
for name in ARMS:
    need = GRAN[name]["nr_med"] / 2.0
    print(f"    {name:>10}  needs cos > {need:>6.3f}   has (p99) {COSD[name]['p99']:>7.4f}   "
          f"{'REACHABLE' if COSD[name]['p99'] > need else 'impossible'}")

    family     N   max cos       p99    median       min
--------------------------------------------------------
      nn-1    53    0.0063    0.0056    0.0005   -0.0102
  random-1   512    0.0184    0.0117    0.0007   -0.0082
     gcg-1   512    0.0182    0.0146    0.0011   -0.0098
  random-7   512    0.0151    0.0099    0.0017   -0.0064
     gcg-7   512    0.0206    0.0131    0.0039   -0.0031

best alignment any token move achieves with the target: cos = 0.0206
for reference, two random vectors in 4096*53 dims: ~0.00215

why the L2 objective can never improve:
  ||Δ - D||^2 = ||Δ||^2 - 2<Δ,D> + ||D||^2
  improving needs  2<Δ,D> > ||Δ||^2,  i.e.  cos > ||Δ|| / (2||D||)
          nn-1  needs cos >  0.329   has (p99)  0.0056   impossible
      random-1  needs cos >  0.613   has (p99)  0.0117   impossible
         gcg-1  needs cos >  0.601   has (p99)  0.0146   impossible
      random-7  needs cos >  1.623   has (p99)  0.0099   impossible
         gcg-7  needs cos >  1.585   has (p99)  

In [29]:
# === save §4 and §5 ===
import json, os
with open("/content/phase7_gradient_target.json") as f:
    out = json.load(f)
out["stage2_match"] = dict(
    layer=8, targets={str(k): v for k, v in RESULTS4.items()},
    note=("GCG minimising ||h_8(T')[slots] - (h_8(TRIG)[slots] + D)||^2; "
          "rel_residual 1.0 = the original trigger, 0.0 = exact hit"))
out["granularity"] = dict(
    target="L8 ascent step 35", D_frob=float(Dn), D_per_slot=float(D.norm(dim=-1).mean()),
    h_nonsink=NSNORM[8], families=GRAN, cosines=COSD,
    note=("||Δ||/||D|| and cos(Δ, D) over the flattened [53,4096] block, for five families "
          "of token move; nn-1 is each slot swapped to its nearest in-pool embedding "
          "neighbour = the finest single move prompt space possesses"))
with open("/content/phase7_gradient_target.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"saved ({os.path.getsize('/content/phase7_gradient_target.json')/1024:.0f} kB)")

saved (100 kB)


In [26]:
# === §4b — the same search, aimed at the DEGENERATE states ===
#
# L24's ascent collapsed the model onto punctuation: step 20 scored 0.0990 emitting only
# commas (distinct 0.02), step 40 scored 0.1041 on '1's. Both sit inside the band phase 6
# calls "real bridge question".
#
# Whether prompt space can reach those states is the sharper question for this project than
# whether it can reach a fluent one. Phase 6's negative is "GCG maximised the objective and
# produced none of the behaviour". If GCG can hit a degenerate state here, the result
# becomes "GCG maximised the objective by destroying the output" — the textbook Goodhart
# shape the project has looked for six times and never actually caught in prompt space.
import torch

L_T = 24                                   # h_slots / grad_match read this global
SNAPS24 = ascend_keep_D(L_T, snap=(20, 40))
for t, D in SNAPS24.items():
    with inject(L_T, D, SLOT0, SLOT1), torch.no_grad():
        a_ = rollout(TRIG, PRE, SUF, 45)
        s_ = score_answer(TRIG, PRE, SUF, a_)
    print(f"reproduced L24/step{t}: true={s_[METRIC_SPACE]:.4f} "
          f"distinct={distinct_ratio(a_):.2f}  {tokenizer.decode(a_, skip_special_tokens=True)[:40]!r}")

H0 = h_slots(TRIG.unsqueeze(0))[0]          # rebuild the reference at L24

for t_step in (20, 40):
    D     = SNAPS24[t_step]
    Hstar = H0 + D
    print(f"\n{'='*96}\nTARGET: L{L_T} ascent step {t_step}  (degenerate)  ||D||/||h|| = "
          f"{(D.norm(dim=-1).mean()/NSNORM[L_T]).item():.3f}")
    print(f"  start: " + "  ".join(f"{k}={v:+.4f}"
                                   for k, v in readout(TRIG, Hstar, D).items()))
    bt, bv, ns, traj = gcg_match(Hstar, D, budget_s=200)
    r = readout(bt, Hstar, D)
    print(f"  after {ns} steps: " + "  ".join(f"{k}={v:+.4f}" for k, v in r.items()))
    with torch.no_grad():
        ans_t = rollout(bt, PRE, SUF, 45)
        sc_t  = score_answer(bt, PRE, SUF, ans_t)
    print(f"  found trigger, NO injection: {METRIC_SPACE}={sc_t[METRIC_SPACE]:.4f} "
          f"distinct={distinct_ratio(ans_t):.2f}   "
          f"(the state itself scored {'0.0990' if t_step==20 else '0.1041'} at distinct "
          f"{'0.02' if t_step==20 else '0.04'})")
    print(f"    {tokenizer.decode(ans_t, skip_special_tokens=True)[:200]!r}")
    RESULTS4[f"L24_{t_step}"] = dict(readout_start=readout(TRIG, Hstar, D), readout_end=r,
                                     steps=ns, trajectory=traj, trigger_ids=bt.tolist(),
                                     trigger=tokenizer.decode(bt), scores=sc_t,
                                     distinct=distinct_ratio(ans_t),
                                     answer=tokenizer.decode(ans_t, skip_special_tokens=True))

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_4700/3800891473.py", line 15, in <cell line: 0>
    SNAPS24 = ascend_keep_D(L_T, snap=(20, 40))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4700/4103685911.py", line 28, in ascend_keep_D
    ans = rollout(TRIG, PRE, SUF, n_new)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4700/159935573.py", line 64, in rollout
    out = model.generate(ids, max_new_tokens=n_new, do_sample=do_sample,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_conte

TypeError: object of type 'NoneType' has no len()